# Demo 07a — Shallow Models and Their Weaknesses

**Class 07 · E1** (slides: `slides/07_ensembles.md`)

A **shallow model** is one fitted model that combines nothing: logistic regression, naive Bayes, one tree, 1-NN, an RBF SVM, a one-layer MLP.
k-NN with $k > 1$ is left out on purpose: it is a vote of $k$ neighbours and belongs to the next demo.

Each model has a data set that breaks it. We look at six planes (frontiers), six function families (curves) and the score of each model on two tables.
Scores: **MCC** for classification, **MAE** for regression, on 3000 test points, mean of five seeds. A red frame marks the best model of a row.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from joblib import Parallel, delayed

from faa import boundaries, demo, metrics, plotlib
from faa import galleries as ga
from faa import synthetic as sy
from faa.demo import show, slide
from faa.galleries import CLASS_SETS, REG_SETS, board
from faa.plotlib import BLUE, GREEN, GREY, ORANGE, RED

demo.setup("07")
SEED = 0
N_JOBS = 8

## The two scores

**MCC** (Matthews correlation coefficient) reads the four counts of the confusion matrix. It is 1 for a perfect classifier, 0 for a constant or random one, and it does not reward a model for predicting the larger class.

$$\text{MCC} = \frac{TP\,TN - FP\,FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

**MAE** (mean absolute error) is the mean of $|y_i - \hat y_i|$, in the units of the target. With Gaussian noise of standard deviation $\sigma$ no model goes below about $0.8\,\sigma$.

In [ ]:
slide("Two Scores: MCC and MAE")
X_tr, y_tr, X_te, y_te = sy.load("skewed blobs", SEED)
tree = ga.build("c", "tree", SEED).fit(X_tr, y_tr)
metric_table = pl.DataFrame(
    [
        ("always negative", float(np.mean(y_te == 0)), metrics.task_score("c", y_te, 0 * y_te)),
        ("tree", float(tree.score(X_te, y_te)), metrics.task_score("c", y_te, tree.predict(X_te))),
    ],
    schema=["model", "accuracy", "MCC"],
    orient="row",
)
print(f"skewed blobs: share of positives in the test set {y_te.mean():.3f}")
show(metric_table)
tp, tn, fp, fn = 30, 50, 10, 10
root = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
print(
    f"by hand: TP {tp}, TN {tn}, FP {fp}, FN {fn}: MCC = ({tp * tn} - {fp * fn}) / {root:.0f} = {(tp * tn - fp * fn) / root:.3f}"
)
err = np.array([0.1, -0.1, 0.1, -0.1, 2.0])
print(f"errors {err.tolist()}: MAE {np.mean(np.abs(err)):.2f}, RMSE {np.sqrt(np.mean(err**2)):.2f}")
print(
    f"ceilings: {sy.FLIP:.0%} flipped labels, MCC {1 - 2 * sy.FLIP:.2f}; noise sd {sy.NOISE}, MAE {sy.NOISE * np.sqrt(2 / np.pi):.3f}"
)

In [ ]:
# plot e1_scores
fg, ax = plotlib.fig("col", 1.3)
x = np.arange(len(metric_table))
ax.bar(x - 0.18, metric_table["accuracy"], width=0.34, color=GREY, label="accuracy")
ax.bar(x + 0.18, metric_table["MCC"], width=0.34, color=BLUE, label="MCC")
for i, (a, m) in enumerate(zip(metric_table["accuracy"], metric_table["MCC"])):
    ax.text(i - 0.18, a + 0.02, f"{a:.2f}", ha="center", fontsize=6.2)
    ax.text(i + 0.18, m + 0.02, f"{m:.2f}", ha="center", fontsize=6.2)
ax.set(xticks=x, xticklabels=["always negative", "tree"], ylim=(0, 1.25), ylabel="score")
ax.grid(axis="x", visible=False)
ax.legend(ncols=2, loc="upper center", fontsize=6.3, handlelength=1.2)
plotlib.save(fg, "e1_scores")

## Six planes, six shallow models

Moons, circles, XOR and the spiral carry 10 % flipped labels: their MCC cannot pass 0.80. One column shows the weakness of one model. The panels show one seed; the board gives the mean of five.

In [ ]:
slide("Shallow Classifiers: the Board")
clf = ga.evaluate(CLASS_SETS, ga.SHALLOW_C)
reg = ga.evaluate(REG_SETS, ga.SHALLOW_R)
show(ga.wide(clf, ga.SHALLOW_C, CLASS_SETS))
show(ga.wide(reg, ga.SHALLOW_R, REG_SETS))

In [ ]:
def sets_figure(sets, names):
    """Rows are data sets, columns are models."""
    task = sy.task_of(sets[0])
    fg, axs = plt.subplots(len(sets), len(names), figsize=(plotlib.FULL, 0.7 * len(sets) + 0.06), squeeze=False)
    fg.subplots_adjust(left=0.035, right=0.998, top=0.93, bottom=0.06, hspace=0.3, wspace=0.05)
    for r, d in enumerate(sets):
        X_tr, y_tr, X_te, y_te = sy.load(d, SEED)
        fitted = dict(zip(names, Parallel(n_jobs=-1)(delayed(ga.build(task, m, SEED).fit)(X_tr, y_tr) for m in names)))
        sc = {m: metrics.task_score(task, y_te, f.predict(X_te)) for m, f in fitted.items()}
        best = (max if task == "c" else min)(sc, key=sc.get)
        lim = boundaries.plane_limits(X_tr, pad=0.25) if task == "c" else None
        ylim = (y_tr.min() - 0.3, y_tr.max() + 0.3)
        for c, m in enumerate(names):
            kw = {
                "title": m if r == 0 else "",
                "label": f"{'MCC' if task == 'c' else 'MAE'} {sc[m]:.2f}",
                "frame": RED if m == best else None,
            }
            if task == "c":
                boundaries.frontier_panel(axs[r, c], fitted[m], X_tr, y_tr, lim, **kw)
            else:
                boundaries.fit_panel(axs[r, c], fitted[m], X_tr, y_tr, sy.CURVES[d], ylim, **kw)
        axs[r, 0].set_ylabel(d, fontsize=6, labelpad=1.5)
    return fg

In [ ]:
plotlib.save(sets_figure(["moons", "circles"], ga.SHALLOW_C), "e1_planes_a")

In [ ]:
plotlib.save(sets_figure(["XOR", "spiral"], ga.SHALLOW_C), "e1_planes_b")

In [ ]:
plotlib.save(sets_figure(["checkerboard", "skewed blobs"], ga.SHALLOW_C), "e1_planes_c")

In [ ]:
plotlib.save(board(clf, "c", CLASS_SETS), "e1_board_c")

## Six function families, six shallow regressors

Orange dots are the 120 training points, the dotted line is the true function, the blue line is the fit. Noise of standard deviation 0.10: no model goes below 0.08.

In [ ]:
plotlib.save(sets_figure(["cubic", "exponential"], ga.SHALLOW_R), "e1_curves_a")

In [ ]:
plotlib.save(sets_figure(["sine", "chirp"], ga.SHALLOW_R), "e1_curves_b")

In [ ]:
plotlib.save(sets_figure(["step", "sawtooth"], ga.SHALLOW_R), "e1_curves_c")

In [ ]:
plotlib.save(board(reg, "r", REG_SETS), "e1_board_r")

## One sample, one model: variance

Refit each model on 30 bootstrap resamples of the spiral: the score of a stable model hardly moves, and a flexible one moves with the sample.

In [ ]:
slide("One Sample, One Model")
from sklearn.base import clone  # noqa: E402
from sklearn.utils import resample  # noqa: E402

X_tr, y_tr, X_te, y_te = sy.load("spiral", SEED)
models = ga.shallow_classifiers(SEED)


def refit(name, k):
    idx = resample(np.arange(len(y_tr)), random_state=k)
    return name, metrics.task_score("c", y_te, clone(models[name]).fit(X_tr[idx], y_tr[idx]).predict(X_te))


runs = Parallel(n_jobs=N_JOBS)(delayed(refit)(n, k) for n in ga.SHALLOW_C for k in range(30))
spread = pl.DataFrame(runs, schema=["model", "MCC"], orient="row").group_by("model", maintain_order=True)
variance = spread.agg(pl.col("MCC").mean().alias("mean MCC"), pl.col("MCC").std().alias("sd"))
show(variance)

In [ ]:
# plot e1_variance
fg, ax = plotlib.fig("col", 1.35)
ax.bar([m.replace(" ", "\n") for m in variance["model"]], variance["sd"], color=BLUE, width=0.65)
for i, v in enumerate(variance["sd"]):
    ax.text(i, v + 0.001, f"{v:.3f}", ha="center", fontsize=5.6)
ax.set(ylabel="sd of the MCC, 30 resamples", ylim=(0, variance["sd"].max() * 1.25))
ax.tick_params(axis="x", labelsize=5.6)
ax.grid(axis="x", visible=False)
plotlib.save(fg, "e1_variance")

## Do the errors overlap?

A combination can only help if the models are wrong on **different** points: count, for each test point of the spiral, how many of the six models get it wrong.

In [ ]:
slide("Do the Errors Overlap?")
preds = np.array([m.fit(X_tr, y_tr).predict(X_te) for m in models.values()])
share = np.bincount((preds != y_te).sum(axis=0), minlength=7) / len(y_te)
show(pl.DataFrame({"models wrong": range(7), "share of test points": share}))
print(
    f"right for all six {share[0]:.3f}; wrong for all six {share[6]:.3f}; the models disagree on {share[1:6].sum():.3f}"
)
print("correlation of the errors:")
show(
    pl.DataFrame(np.corrcoef((preds != y_te).astype(float)), schema=list(models)).insert_column(
        0, pl.Series("model", list(models))
    ),
    2,
)

In [ ]:
# plot e1_overlap
fg, ax = plotlib.fig("col", 1.35)
ax.bar(np.arange(7), share, color=[GREEN, *[ORANGE] * 5, RED], width=0.7)
for k, s in enumerate(share):
    ax.text(k, s + 0.008, f"{s:.2f}", ha="center", fontsize=6)
ax.set(
    xlabel="models wrong on a test point (of 6)",
    ylabel="share of test points",
    ylim=(0, share.max() * 1.25),
    xticks=np.arange(7),
)
ax.grid(axis="x", visible=False)
plotlib.save(fg, "e1_overlap")

### Sweep and failure case: more label noise (no slide)

Flipping more labels hurts the models that memorise points first: 1-NN copies each flipped label, the smooth SVM averages them.

In [ ]:
slide("Label Noise Against the Shallow Models (no slide)")
from sklearn.datasets import make_moons  # noqa: E402


def flip_run(flip, name, seed):
    X, y = make_moons(3600, noise=0.15, random_state=seed)
    y = sy.flip_labels(y, flip, seed)
    return (
        flip,
        name,
        metrics.task_score("c", y[600:], ga.build("c", name, seed).fit(X[:600], y[:600]).predict(X[600:])),
    )


rows = Parallel(n_jobs=N_JOBS)(
    delayed(flip_run)(f, n, s) for f in (0.0, 0.1, 0.2, 0.3) for n in ga.SHALLOW_C for s in range(5)
)
noise = (
    pl.DataFrame(rows, schema=["flip", "model", "MCC"], orient="row")
    .group_by("flip", "model")
    .agg(pl.col("MCC").mean())
)
show(noise.pivot(on="model", index="flip", values="MCC").sort("flip"))